# 02. Exploratory Data Analysis (EDA)

In this notebook, we perform deep exploratory analysis on the multi-city accommodation dataset to inform feature engineering and retrieval strategies.

### Key Insights Explored:
1. **Volume & Density**: Listing distribution across cities.
2. **Price Distribution**: Price percentiles, outliers, and variance by city & room type.
3. **Ratings & Review Dynamics**: Correlation between rating scores and review volume.
4. **Amenities Landscape**: Most prevalent amenities and their frequency across regions.
5. **Spatial Distribution**: Geographic coordinate spreads.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import config

# Styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load processed listings
df = pd.read_parquet(config.LISTINGS_CLEAN_PATH)
print(f"Loaded dataset: {df.shape[0]:,} listings, {df.shape[1]} features")

## 1. City Listing Breakdown

In [ ]:
city_counts = df['city'].value_counts()
plt.figure(figsize=(10, 5))
sns.barplot(x=city_counts.index, y=city_counts.values, palette="viridis")
plt.title("Listings Distribution by City", fontsize=14, fontweight="bold")
plt.ylabel("Number of Listings")
plt.xlabel("City")
for i, v in enumerate(city_counts.values):
    plt.text(i, v + 100, f"{v:,}", ha="center", fontweight="bold")
plt.show()

## 2. Price Distribution Across Cities (Log Scale & Boxplot)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df[df['price_usd'] <= 600], x='city', y='price_usd', hue='room_type', palette="Set2")
plt.title("Nightly Price ($) by City and Room Type (Capped at $600)", fontsize=14, fontweight="bold")
plt.ylabel("Price ($ USD)")
plt.xlabel("City")
plt.legend(title="Room Type", loc="upper right")
plt.show()

## 3. Rating Scores & Review Volume

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['review_scores_rating'].dropna(), bins=30, kde=True, ax=axes[0], color="royalblue")
axes[0].set_title("Distribution of Review Rating Scores", fontweight="bold")
axes[0].set_xlabel("Rating Score (0 - 5)")

sns.histplot(np.log1p(df['number_of_reviews'].dropna()), bins=30, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Distribution of log(1 + Number of Reviews)", fontweight="bold")
axes[1].set_xlabel("log(1 + Reviews)")

plt.tight_layout()
plt.show()

## 4. Top Amenities Frequency

In [ ]:
amenity_cols = [c for c in df.columns if c.startswith('amenity_')]
amenity_freq = df[amenity_cols].mean().sort_values(ascending=False).head(15)
amenity_labels = [c.replace('amenity_', '').replace('_', ' ').title() for c in amenity_freq.index]

plt.figure(figsize=(12, 6))
sns.barplot(x=amenity_freq.values, y=amenity_labels, palette="mako")
plt.title("Top 15 Most Prevalent Amenities Across Listings", fontsize=14, fontweight="bold")
plt.xlabel("Proportion of Listings with Amenity")
plt.xlim(0, 1.0)
for i, v in enumerate(amenity_freq.values):
    plt.text(v + 0.01, i, f"{v:.1%}", va="center", fontweight="bold")
plt.show()